# FLOPs-Budgeted HPO for Sampled NATS Architectures

This notebook runs real CIFAR-10 training for architectures from `sampled_architectures_10.jsonl` under a per-architecture FLOPs budget. FLOPs and parameter counts are read only from `sampled_architecture_costs.csv`.

In [1]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q torch torchvision pandas plotly numpy
%pip install -q xautodl --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [2]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms as T

from xautodl.config_utils import dict2config
from xautodl.models import get_cell_based_tiny_net

## Settings

Edit only this cell for the main experiment knobs: hyperparameter values, scheduler choices, and the FLOPs budget.

In [34]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "sampled_architectures_10.jsonl").exists():
            return candidate
    raise FileNotFoundError("Could not find sampled_architectures_10.jsonl in cwd parents")


REPO_ROOT = find_repo_root()
ARCHITECTURES_PATH = REPO_ROOT / "sampled_architectures_10.jsonl"
COSTS_CSV_PATH = REPO_ROOT / "sampled_architecture_costs.csv"
OUTPUT_DIR = REPO_ROOT / "hpo_output"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

# Architecture subset. Use None for all rows from sampled_architectures_10.jsonl.
ARCH_ROWS: list[int] | None = None

# FLOPs budget per architecture, not for the whole notebook.
BUDGET_FLOPS_PER_ARCH = 1.0e15

# ASHA schedule targets. The effective values are computed per architecture from the FLOPs budget.
TARGET_MIN_EPOCHS = 3
TARGET_REDUCTION_FACTOR = 3
MIN_INITIAL_CONFIGS = 1
MAX_INITIAL_CONFIGS: int | None = 12  # None means the FLOPs budget decides how many initial configs fit.
MAX_EPOCHS_CAP: int | None = 81  # None means ASHA can use as many epochs as fit in the FLOPs budget.

# HPO search space. lr and weight_decay are sampled log-uniformly per trial.
LR_RANGE = (1e-3, 3e-1)
WEIGHT_DECAY_RANGE = (1e-6, 1e-3)

# Training settings kept fixed across HPO trials.
BATCH_SIZE = 256
NUM_WORKERS = 2
VALIDATION_FRACTION = 0.1
MOMENTUM = 0.9
GRAD_CLIP_NORM: float | None = 5.0
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")

Repo root: /content
Device: cuda


## Inputs

`sampled_architecture_costs.csv` must contain at least: `arch_row`, `arch_index`, `dataset`, `forward_flops_per_sample`.

In [35]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for row_index, line in enumerate(f):
            record = json.loads(line)
            record["arch_row"] = row_index
            record["arch_index"] = int(record["arch_index"])
            records.append(record)
    if ARCH_ROWS is not None:
        keep = set(ARCH_ROWS)
        records = [record for record in records if record["arch_row"] in keep]
    if not records:
        raise ValueError("No architectures selected")
    return records


def load_costs(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Generate it once from NATS-Bench before running real HPO."
        )
    costs = pd.read_csv(path)
    required = {"arch_row", "arch_index", "dataset", "forward_flops_per_sample"}
    missing = required - set(costs.columns)
    if missing:
        raise ValueError(f"Costs CSV is missing columns: {sorted(missing)}")
    return costs


arch_records = load_jsonl(ARCHITECTURES_PATH)
arch_df = pd.DataFrame(arch_records)
costs_df = load_costs(COSTS_CSV_PATH)
display(arch_df)
display(costs_df.head())

,search_space,dataset,arch_index,arch_str,arch_row
0,tss,cifar10-valid,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,0


,arch_row,arch_index,dataset,forward_flops_per_sample,params,latency,raw_flops_m
0,0,13197,cifar10-valid,47104650.0,0.344346,0.015939,47.10465
1,1,3358,cifar10-valid,19579530.0,0.157306,0.017312,19.57953
2,2,11898,cifar10-valid,51036810.0,0.372346,0.018143,51.03681
3,3,11570,cifar10-valid,47104650.0,0.344346,0.015434,47.10465
4,4,8712,cifar10-valid,11715210.0,0.101306,0.012853,11.71521


## Dataset

The split and transforms are fixed for all trials.

In [36]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


normalize_kwargs = {
    "mean": [0.49139968, 0.48215841, 0.44653091],
    "std": [0.24703223, 0.24348513, 0.26158784],
}

train_transform = T.Compose([
    T.RandomCrop(size=32, padding=4),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(**normalize_kwargs),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(**normalize_kwargs),
])

data_root = REPO_ROOT / "data"
train_aug_dataset = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_transform)
train_eval_dataset = datasets.CIFAR10(root=data_root, train=True, download=True, transform=eval_transform)
test_dataset = datasets.CIFAR10(root=data_root, train=False, download=True, transform=eval_transform)

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(train_aug_dataset), generator=generator).tolist()
n_val = int(round(len(indices) * VALIDATION_FRACTION))
val_indices = indices[:n_val]
train_indices = indices[n_val:]
N_TRAIN_EXAMPLES = len(train_indices)

train_loader = DataLoader(
    Subset(train_aug_dataset, train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    Subset(train_eval_dataset, val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train examples: {N_TRAIN_EXAMPLES}")
print(f"Validation examples: {len(val_indices)}")
print(f"Test examples: {len(test_dataset)}")

Train examples: 45000
Validation examples: 5000
Test examples: 10000


## FLOPs Accounting

Training FLOPs are estimated as `3 * forward_flops_per_sample * train_examples * epochs`. Validation FLOPs are counted as `forward_flops_per_sample * validation_examples` after every evaluated ASHA stage. The CSV is the only source of architecture costs.

In [37]:
costs_by_row = costs_df.set_index("arch_row").to_dict(orient="index")


def get_arch_flops_record(arch_record: dict[str, Any]) -> dict[str, Any]:
    row = costs_by_row[arch_record["arch_row"]]
    forward_flops_per_sample = int(row["forward_flops_per_sample"])
    epoch_flops = int(3 * forward_flops_per_sample * N_TRAIN_EXAMPLES)
    validation_flops = int(forward_flops_per_sample * len(val_indices))
    return {
        "arch_row": arch_record["arch_row"],
        "arch_index": arch_record["arch_index"],
        "dataset": arch_record["dataset"],
        "forward_flops_per_sample": forward_flops_per_sample,
        "raw_flops_m": row.get("raw_flops_m", forward_flops_per_sample / 1_000_000),
        "params": row.get("params", np.nan),
        "latency": row.get("latency", np.nan),
        "n_train_examples": N_TRAIN_EXAMPLES,
        "n_validation_examples": len(val_indices),
        "epoch_flops": epoch_flops,
        "validation_flops": validation_flops,
        "budget_flops": BUDGET_FLOPS_PER_ARCH,
        "budget_in_epochs_for_one_config": BUDGET_FLOPS_PER_ARCH / epoch_flops,
    }


arch_flops_df = pd.DataFrame([get_arch_flops_record(record) for record in arch_records])
display(arch_flops_df)

,arch_row,arch_index,dataset,forward_flops_per_sample,raw_flops_m,params,latency,n_train_examples,n_validation_examples,epoch_flops,validation_flops,budget_flops,budget_in_epochs_for_one_config
0,0,13197,cifar10-valid,47104650,47.10465,0.344346,0.015939,45000,5000,6359127750000,235523250000,1.000000e+15,157.254271


## HPO Configuration

In [38]:
@dataclass(frozen=True)
class HPOConfig:
    trial_id: int
    lr: float
    weight_decay: float


@dataclass(frozen=True)
class ASHAPlan:
    min_epochs: int
    max_epochs: int
    reduction_factor: int
    num_initial_configs: int
    budget_epochs: int
    rungs: list[int]
    planned_train_epochs: int
    planned_validation_stages: int
    planned_flops: int


def sample_log_uniform(rng: random.Random, low: float, high: float) -> float:
    return math.exp(rng.uniform(math.log(low), math.log(high)))


def build_hpo_configs(num_initial_configs: int, seed: int) -> list[HPOConfig]:
    rng = random.Random(seed)
    configs = []
    for trial_id in range(num_initial_configs):
        configs.append(
            HPOConfig(
                trial_id=trial_id,
                lr=sample_log_uniform(rng, *LR_RANGE),
                weight_decay=sample_log_uniform(rng, *WEIGHT_DECAY_RANGE),
            )
        )
    return configs



## Real Training

In [39]:
def create_nats_model(arch_str: str, num_classes: int = 10) -> nn.Module:
    config = dict2config(
        {
            "name": "infer.tiny",
            "C": 16,
            "N": 5,
            "arch_str": arch_str,
            "num_classes": num_classes,
        },
        None,
    )
    return get_cell_based_tiny_net(config)


def create_optimizer_and_scheduler(model: nn.Module, config: HPOConfig, schedule_max_epochs: int):
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=config.lr,
        momentum=MOMENTUM,
        weight_decay=config.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=schedule_max_epochs)
    return optimizer, scheduler


def accuracy_top1(logits: torch.Tensor, targets: torch.Tensor) -> float:
    predictions = logits.argmax(dim=1)
    return float((predictions == targets).sum().item())


def extract_logits(model_output: Any) -> torch.Tensor:
    if torch.is_tensor(model_output):
        return model_output
    if isinstance(model_output, (tuple, list)):
        tensors = [item for item in model_output if torch.is_tensor(item)]
        if tensors:
            return tensors[-1]
    raise TypeError(f"Model output does not contain logits tensor: {type(model_output)!r}")


def train_one_epoch(model: nn.Module, optimizer, criterion, loader: DataLoader) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = extract_logits(model(images))
        loss = criterion(logits, targets)
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        batch_size = int(targets.size(0))
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
    return total_loss / max(1, total_examples)


@torch.inference_mode()
def evaluate_top1(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_correct = 0.0
    total_examples = 0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits = extract_logits(model(images))
        total_correct += accuracy_top1(logits, targets)
        total_examples += int(targets.size(0))
    return 100.0 * total_correct / max(1, total_examples)


def checkpoint_path(arch_row: int, trial_id: int) -> Path:
    return CHECKPOINT_DIR / f"arch_{arch_row:02d}_trial_{trial_id:02d}.pt"


def stage_checkpoint_path(arch_row: int, trial_id: int, target_epochs: int) -> Path:
    return CHECKPOINT_DIR / f"arch_{arch_row:02d}_trial_{trial_id:02d}_epoch_{target_epochs:04d}.pt"


def load_model_from_checkpoint(checkpoint_file: str | Path) -> nn.Module:
    checkpoint = torch.load(checkpoint_file, map_location=DEVICE)
    model = create_nats_model(checkpoint["arch_record"]["arch_str"]).to(DEVICE)
    model.load_state_dict(checkpoint["model"])
    return model


def run_real_training_stage(config: HPOConfig, target_epochs: int, schedule_max_epochs: int, seed: int, arch_record: dict[str, Any]) -> dict[str, Any]:
    set_seed(seed)
    model = create_nats_model(arch_record["arch_str"]).to(DEVICE)
    optimizer, scheduler = create_optimizer_and_scheduler(model, config, schedule_max_epochs)
    criterion = nn.CrossEntropyLoss()

    ckpt_path = checkpoint_path(arch_record["arch_row"], config.trial_id)
    stage_ckpt_path = stage_checkpoint_path(arch_record["arch_row"], config.trial_id, target_epochs)
    start_epoch = 0
    last_val_acc1 = np.nan
    if ckpt_path.exists():
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        if checkpoint.get("schedule_max_epochs") == schedule_max_epochs:
            model.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            start_epoch = int(checkpoint["epoch"])
            last_val_acc1 = float(checkpoint.get("val_acc1", np.nan))
        else:
            print(f"Ignoring checkpoint with different schedule_max_epochs: {ckpt_path}")

    if start_epoch >= target_epochs and not math.isnan(last_val_acc1):
        return {"val_acc1": last_val_acc1, "checkpoint_path": str(stage_ckpt_path if stage_ckpt_path.exists() else ckpt_path)}

    started_at = time.time()
    for epoch in range(start_epoch, target_epochs):
        train_loss = train_one_epoch(model, optimizer, criterion, train_loader)
        scheduler.step()
        print(
            f"arch_row={arch_record['arch_row']} trial={config.trial_id} "
            f"epoch={epoch + 1}/{target_epochs} loss={train_loss:.4f} "
            f"lr={optimizer.param_groups[0]['lr']:.3e}"
        )

    val_acc1 = evaluate_top1(model, val_loader)
    checkpoint_payload = {
            "epoch": target_epochs,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "schedule_max_epochs": schedule_max_epochs,
            "val_acc1": val_acc1,
            "config": asdict(config),
            "arch_record": arch_record,
        }
    torch.save(checkpoint_payload, ckpt_path)
    torch.save(checkpoint_payload, stage_ckpt_path)
    print(
        f"arch_row={arch_record['arch_row']} trial={config.trial_id} "
        f"target_epochs={target_epochs} val_acc1={val_acc1:.2f} "
        f"elapsed_min={(time.time() - started_at) / 60:.1f}"
    )
    return {"val_acc1": val_acc1, "checkpoint_path": str(stage_ckpt_path)}

## ASHA Under FLOPs Budget

In [40]:
def asha_rungs(min_epochs: int, max_epochs: int, reduction_factor: int) -> list[int]:
    rungs = []
    current = min_epochs
    while current < max_epochs:
        rungs.append(current)
        current *= reduction_factor
    rungs.append(max_epochs)
    return sorted(set(rungs))


def estimate_asha_work(num_configs: int, rungs: list[int], reduction_factor: int) -> tuple[int, int]:
    total_epochs = 0
    total_stages = 0
    alive = num_configs
    completed_epochs = 0
    for target_epochs in rungs:
        total_stages += alive
        total_epochs += alive * (target_epochs - completed_epochs)
        alive = max(1, math.ceil(alive / reduction_factor))
        completed_epochs = target_epochs
    return total_epochs, total_stages


def estimate_asha_flops(num_configs: int, rungs: list[int], reduction_factor: int, epoch_flops: int, validation_flops: int) -> tuple[int, int, int]:
    train_epochs, validation_stages = estimate_asha_work(num_configs, rungs, reduction_factor)
    total_flops = train_epochs * epoch_flops + validation_stages * validation_flops
    return train_epochs, validation_stages, total_flops


def make_asha_plan(budget_flops: float, epoch_flops: int, validation_flops: int) -> ASHAPlan:
    budget_epochs = int(budget_flops // epoch_flops)
    if budget_epochs < 1:
        raise ValueError(f"Budget is smaller than one training epoch: budget_flops={budget_flops}, epoch_flops={epoch_flops}")

    reduction_factor = max(2, TARGET_REDUCTION_FACTOR)
    min_epochs = max(1, min(TARGET_MIN_EPOCHS, budget_epochs))
    min_stage_flops = min_epochs * epoch_flops + validation_flops
    max_configs_that_fit_min_stage = max(1, int(budget_flops // min_stage_flops))
    if MAX_INITIAL_CONFIGS is None:
        num_initial_configs = max_configs_that_fit_min_stage
    else:
        num_initial_configs = min(MAX_INITIAL_CONFIGS, max_configs_that_fit_min_stage)
    num_initial_configs = max(MIN_INITIAL_CONFIGS, num_initial_configs)
    rungs = [min_epochs]
    while True:
        next_epoch = rungs[-1] * reduction_factor
        if MAX_EPOCHS_CAP is not None and next_epoch > MAX_EPOCHS_CAP:
            break
        candidate_rungs = rungs + [next_epoch]
        _, _, candidate_flops = estimate_asha_flops(
            num_initial_configs,
            candidate_rungs,
            reduction_factor,
            epoch_flops,
            validation_flops,
        )
        if candidate_flops > budget_flops:
            break
        rungs = candidate_rungs

    planned_train_epochs, planned_validation_stages, planned_flops = estimate_asha_flops(
        num_initial_configs,
        rungs,
        reduction_factor,
        epoch_flops,
        validation_flops,
    )
    return ASHAPlan(
        min_epochs=min_epochs,
        max_epochs=rungs[-1],
        reduction_factor=reduction_factor,
        num_initial_configs=num_initial_configs,
        budget_epochs=budget_epochs,
        rungs=rungs,
        planned_train_epochs=planned_train_epochs,
        planned_validation_stages=planned_validation_stages,
        planned_flops=planned_flops,
    )


def run_asha_with_flops_budget(
    arch_record: dict[str, Any],
    budget_flops: float,
    epoch_flops: int,
    validation_flops: int,
    seed: int,
) -> pd.DataFrame:
    plan = make_asha_plan(budget_flops, epoch_flops, validation_flops)
    configs = build_hpo_configs(plan.num_initial_configs, seed)
    spent_flops = 0
    spent_train_flops = 0
    spent_validation_flops = 0
    completed_epochs = {config.trial_id: 0 for config in configs}
    alive = list(configs)
    records: list[dict[str, Any]] = []

    print(
        f"arch_row={arch_record['arch_row']} ASHA plan: "
        f"configs={plan.num_initial_configs}, rungs={plan.rungs}, "
        f"planned_train_epochs={plan.planned_train_epochs}, "
        f"planned_validation_stages={plan.planned_validation_stages}, "
        f"planned_flops={plan.planned_flops:.3e}/{budget_flops:.3e}"
    )

    for rung_id, target_epochs in enumerate(plan.rungs):
        rung_records = []
        for config in alive:
            incremental_epochs = target_epochs - completed_epochs[config.trial_id]
            if incremental_epochs <= 0:
                continue
            train_flops = incremental_epochs * epoch_flops
            stage_flops = train_flops + validation_flops
            if spent_flops + stage_flops > budget_flops:
                records.append({
                    **asdict(config),
                    "rung": rung_id,
                    "target_epochs": target_epochs,
                    "incremental_epochs": incremental_epochs,
                    "incremental_flops": stage_flops,
                    "train_flops": train_flops,
                    "validation_flops": validation_flops,
                    "stage_flops": stage_flops,
                    "cumulative_flops": spent_flops,
                    "cumulative_train_flops": spent_train_flops,
                    "cumulative_validation_flops": spent_validation_flops,
                    "val_acc1": np.nan,
                    "checkpoint_path": np.nan,
                    "status": "skipped_budget_exhausted",
                    "asha_min_epochs": plan.min_epochs,
                    "asha_max_epochs": plan.max_epochs,
                    "asha_reduction_factor": plan.reduction_factor,
                    "asha_num_initial_configs": plan.num_initial_configs,
                    "asha_budget_epochs": plan.budget_epochs,
                    "asha_planned_train_epochs": plan.planned_train_epochs,
                    "asha_planned_validation_stages": plan.planned_validation_stages,
                    "asha_planned_flops": plan.planned_flops,
                })
                continue

            stage_result = run_real_training_stage(config, target_epochs, plan.max_epochs, seed, arch_record)
            val_acc1 = float(stage_result["val_acc1"])
            checkpoint_file = stage_result["checkpoint_path"]
            spent_flops += stage_flops
            spent_train_flops += train_flops
            spent_validation_flops += validation_flops
            completed_epochs[config.trial_id] = target_epochs
            record = {
                **asdict(config),
                "rung": rung_id,
                "target_epochs": target_epochs,
                "incremental_epochs": incremental_epochs,
                "incremental_flops": stage_flops,
                "train_flops": train_flops,
                "validation_flops": validation_flops,
                "stage_flops": stage_flops,
                "cumulative_flops": spent_flops,
                "cumulative_train_flops": spent_train_flops,
                "cumulative_validation_flops": spent_validation_flops,
                "val_acc1": val_acc1,
                "checkpoint_path": checkpoint_file,
                "status": "completed",
                "asha_min_epochs": plan.min_epochs,
                "asha_max_epochs": plan.max_epochs,
                "asha_reduction_factor": plan.reduction_factor,
                "asha_num_initial_configs": plan.num_initial_configs,
                "asha_budget_epochs": plan.budget_epochs,
                "asha_planned_train_epochs": plan.planned_train_epochs,
                "asha_planned_validation_stages": plan.planned_validation_stages,
                "asha_planned_flops": plan.planned_flops,
            }
            records.append(record)
            rung_records.append(record)

        if not rung_records:
            break
        rung_df = pd.DataFrame(rung_records).sort_values("val_acc1", ascending=False)
        n_promoted = max(1, math.ceil(len(rung_df) / plan.reduction_factor))
        promoted_ids = set(rung_df.head(n_promoted)["trial_id"])
        alive = [config for config in alive if config.trial_id in promoted_ids]

    result = pd.DataFrame(records)
    completed = result[result["status"] == "completed"].copy()
    if not completed.empty:
        result.loc[completed.index, "best_val_acc1_so_far"] = completed["val_acc1"].cummax()
    return result


arch_flops_by_row = arch_flops_df.set_index("arch_row").to_dict(orient="index")


def build_hpo_plan_preview() -> tuple[pd.DataFrame, pd.DataFrame]:
    plan_rows = []
    config_rows = []
    for arch_record in arch_records:
        flops_record = arch_flops_by_row[arch_record["arch_row"]]
        plan = make_asha_plan(
            BUDGET_FLOPS_PER_ARCH,
            int(flops_record["epoch_flops"]),
            int(flops_record["validation_flops"]),
        )
        plan_rows.append({
            "arch_row": arch_record["arch_row"],
            "arch_index": arch_record["arch_index"],
            "num_initial_configs": plan.num_initial_configs,
            "rungs": plan.rungs,
            "planned_train_epochs": plan.planned_train_epochs,
            "planned_validation_stages": plan.planned_validation_stages,
            "planned_flops": plan.planned_flops,
            "budget_flops": BUDGET_FLOPS_PER_ARCH,
            "planned_budget_pct": 100.0 * plan.planned_flops / BUDGET_FLOPS_PER_ARCH,
        })

        configs = build_hpo_configs(plan.num_initial_configs, SEED + arch_record["arch_row"])
        for config in configs:
            config_rows.append({
                "arch_row": arch_record["arch_row"],
                "arch_index": arch_record["arch_index"],
                "trial_id": config.trial_id,
                "lr": config.lr,
                "weight_decay": config.weight_decay,
            })
    return pd.DataFrame(plan_rows), pd.DataFrame(config_rows)


hpo_plan_preview_df, hpo_config_preview_df = build_hpo_plan_preview()
display(hpo_plan_preview_df)
display(hpo_config_preview_df)


def run_hpo_for_architecture(arch_record: dict[str, Any]) -> pd.DataFrame:
    flops_record = arch_flops_by_row[arch_record["arch_row"]]
    result = run_asha_with_flops_budget(
        arch_record=arch_record,
        budget_flops=BUDGET_FLOPS_PER_ARCH,
        epoch_flops=int(flops_record["epoch_flops"]),
        validation_flops=int(flops_record["validation_flops"]),
        seed=SEED + arch_record["arch_row"],
    )
    result.insert(0, "arch_row", arch_record["arch_row"])
    result.insert(1, "arch_index", arch_record["arch_index"])
    result.insert(2, "arch_str", arch_record["arch_str"])
    result.insert(3, "dataset", arch_record["dataset"])
    result["search_space"] = arch_record["search_space"]
    for key in ["forward_flops_per_sample", "raw_flops_m", "params", "latency", "epoch_flops", "validation_flops", "budget_flops"]:
        result[key] = flops_record[key]
    return result


all_results = pd.concat([run_hpo_for_architecture(record) for record in arch_records], ignore_index=True)
display(all_results)

,arch_row,arch_index,num_initial_configs,rungs,planned_train_epochs,planned_validation_stages,planned_flops,budget_flops,planned_budget_pct
0,0,13197,12,"[3, 9, 27, 81]",150,19,958344104250000,1.000000e+15,95.83441


,arch_row,arch_index,trial_id,lr,weight_decay
0,0,13197,0,0.038365,0.000001
1,0,13197,1,0.004800,0.000005
2,0,13197,2,0.066731,0.000107
3,0,13197,3,0.162195,0.000002
4,0,13197,4,0.011096,0.000001
5,0,13197,5,0.003480,0.000033
6,0,13197,6,0.001163,0.000004
7,0,13197,7,0.040723,0.000043
8,0,13197,8,0.003516,0.000059
9,0,13197,9,0.101171,0.000001


arch_row=0 ASHA plan: configs=12, rungs=[3, 9, 27, 81], planned_train_epochs=150, planned_validation_stages=19, planned_flops=9.583e+14/1.000e+15
Ignoring checkpoint with different schedule_max_epochs: /content/hpo_output/checkpoints/arch_00_trial_00.pt
arch_row=0 trial=0 epoch=1/3 loss=1.6681 lr=3.835e-02
arch_row=0 trial=0 epoch=2/3 loss=1.3462 lr=3.831e-02
arch_row=0 trial=0 epoch=3/3 loss=1.1459 lr=3.824e-02
arch_row=0 trial=0 target_epochs=3 val_acc1=55.60 elapsed_min=1.1
Ignoring checkpoint with different schedule_max_epochs: /content/hpo_output/checkpoints/arch_00_trial_01.pt
arch_row=0 trial=1 epoch=1/3 loss=1.7931 lr=4.799e-03
arch_row=0 trial=1 epoch=2/3 loss=1.4082 lr=4.793e-03
arch_row=0 trial=1 epoch=3/3 loss=1.2194 lr=4.784e-03
arch_row=0 trial=1 target_epochs=3 val_acc1=53.66 elapsed_min=1.1
Ignoring checkpoint with different schedule_max_epochs: /content/hpo_output/checkpoints/arch_00_trial_02.pt
arch_row=0 trial=2 epoch=1/3 loss=1.7068 lr=6.671e-02
arch_row=0 trial=2 e

,arch_row,arch_index,arch_str,dataset,trial_id,lr,weight_decay,rung,target_epochs,incremental_epochs,...,asha_planned_validation_stages,asha_planned_flops,best_val_acc1_so_far,search_space,forward_flops_per_sample,raw_flops_m,params,latency,epoch_flops,budget_flops
0,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,0,0.038365,0.000001,0,3,3,...,19,958344104250000,55.60,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
1,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,1,0.004800,0.000005,0,3,3,...,19,958344104250000,55.60,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
2,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,2,0.066731,0.000107,0,3,3,...,19,958344104250000,57.18,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
3,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,3,0.162195,0.000002,0,3,3,...,19,958344104250000,57.18,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
4,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,4,0.011096,0.000001,0,3,3,...,19,958344104250000,57.18,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
5,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,5,0.003480,0.000033,0,3,3,...,19,958344104250000,57.18,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
6,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,6,0.001163,0.000004,0,3,3,...,19,958344104250000,57.18,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
7,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,7,0.040723,0.000043,0,3,3,...,19,958344104250000,60.00,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
8,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,8,0.003516,0.000059,0,3,3,...,19,958344104250000,60.00,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15
9,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,9,0.101171,0.000001,0,3,3,...,19,958344104250000,60.00,tss,47104650,47.10465,0.344346,0.015939,6359127750000,1.000000e+15


## Results

In [41]:
completed = all_results[all_results["status"] == "completed"].copy()

if completed.empty:
    arch_summary = pd.DataFrame()
else:
    arch_summary = (
        completed.sort_values(["arch_row", "val_acc1"])
        .groupby("arch_row", as_index=False)
        .tail(1)
        .sort_values("val_acc1", ascending=False)
        .copy()
    )
    spent_by_arch = completed.groupby("arch_row")["incremental_flops"].sum()
    train_spent_by_arch = completed.groupby("arch_row")["train_flops"].sum()
    validation_spent_by_arch = completed.groupby("arch_row")["validation_flops"].sum()
    stages_by_arch = completed.groupby("arch_row").size()
    arch_summary["spent_flops"] = arch_summary["arch_row"].map(spent_by_arch)
    arch_summary["spent_train_flops"] = arch_summary["arch_row"].map(train_spent_by_arch)
    arch_summary["spent_validation_flops"] = arch_summary["arch_row"].map(validation_spent_by_arch)
    arch_summary["spent_budget_ratio"] = arch_summary["spent_flops"] / BUDGET_FLOPS_PER_ARCH
    arch_summary["completed_stages"] = arch_summary["arch_row"].map(stages_by_arch)

    test_rows = []
    for _, row in arch_summary.iterrows():
        checkpoint_file = row.get("checkpoint_path")
        if not isinstance(checkpoint_file, str) or not Path(checkpoint_file).exists():
            test_rows.append({"arch_row": row["arch_row"], "test_acc1": np.nan, "test_flops": np.nan})
            continue
        model = load_model_from_checkpoint(checkpoint_file)
        test_acc1 = evaluate_top1(model, test_loader)
        test_flops = int(row["forward_flops_per_sample"] * len(test_dataset))
        test_rows.append({"arch_row": row["arch_row"], "test_acc1": test_acc1, "test_flops": test_flops})
    test_df = pd.DataFrame(test_rows).set_index("arch_row")
    arch_summary["test_acc1"] = arch_summary["arch_row"].map(test_df["test_acc1"])
    arch_summary["test_flops"] = arch_summary["arch_row"].map(test_df["test_flops"])

display(arch_summary)

results_path = OUTPUT_DIR / "hpo_real_training_results.csv"
summary_path = OUTPUT_DIR / "hpo_real_training_summary.csv"
all_results.to_csv(results_path, index=False)
arch_summary.to_csv(summary_path, index=False)
print(f"Saved results to {results_path}")
print(f"Saved summary to {summary_path}")

,arch_row,arch_index,arch_str,dataset,trial_id,lr,weight_decay,rung,target_epochs,incremental_epochs,...,latency,epoch_flops,budget_flops,spent_flops,spent_train_flops,spent_validation_flops,spent_budget_ratio,completed_stages,test_acc1,test_flops
18,0,13197,|nor_conv_3x3~0|+|avg_pool_3x3~0|avg_pool_3x3~...,cifar10-valid,11,0.006964,0.000003,3,81,54,...,0.015939,6359127750000,1.000000e+15,958344104250000,953869162500000,4474941750000,0.958344,19,86.71,471046500000


Saved results to /content/hpo_output/hpo_real_training_results.csv
Saved summary to /content/hpo_output/hpo_real_training_summary.csv


In [42]:
if not completed.empty:
    plot_df = completed.sort_values(["arch_row", "cumulative_flops"]).copy()
    plot_df["arch_label"] = "row=" + plot_df["arch_row"].astype(str) + ", idx=" + plot_df["arch_index"].astype(str)
    plot_df["trial_label"] = "trial " + plot_df["trial_id"].astype(str)
    plot_df["budget_used_pct"] = 100.0 * plot_df["cumulative_flops"] / plot_df["budget_flops"]
    plot_df["lr_text"] = plot_df["lr"].map(lambda value: f"{value:.2e}")
    plot_df["weight_decay_text"] = plot_df["weight_decay"].map(lambda value: f"{value:.2e}")

    if not arch_summary.empty:
        overview_df = arch_summary.copy()
        overview_df["arch_label"] = "row=" + overview_df["arch_row"].astype(str) + ", idx=" + overview_df["arch_index"].astype(str)
        overview_df["budget_used_pct"] = 100.0 * overview_df["spent_budget_ratio"]
        fig_overview = px.bar(
            overview_df.sort_values("val_acc1", ascending=False),
            x="arch_label",
            y="val_acc1",
            color="budget_used_pct",
            text="val_acc1",
            hover_data=["trial_id", "target_epochs", "lr", "weight_decay", "test_acc1", "spent_flops", "spent_train_flops", "spent_validation_flops", "budget_used_pct"],
            title="Best HPO result by architecture",
            color_continuous_scale="Viridis",
        )
        fig_overview.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig_overview.update_xaxes(title="Architecture")
        fig_overview.update_yaxes(title="Best validation Acc@1", rangemode="tozero")
        fig_overview.update_layout(height=520)
        fig_overview.show()

    for arch_row, arch_df in plot_df.groupby("arch_row", sort=True):
        arch_df = arch_df.sort_values("cumulative_flops").copy()
        arch_index = int(arch_df["arch_index"].iloc[0])
        arch_title = f"Architecture row={arch_row}, index={arch_index}"

        best_stage = arch_df.loc[arch_df["val_acc1"].idxmax()]
        print(
            f"{arch_title}: best Acc@1={best_stage['val_acc1']:.2f}, "
            f"trial={int(best_stage['trial_id'])}, epochs={int(best_stage['target_epochs'])}, "
            f"lr={best_stage['lr']:.2e}, wd={best_stage['weight_decay']:.2e}"
        )

        fig_progress = px.line(
            arch_df,
            x="budget_used_pct",
            y="best_val_acc1_so_far",
            markers=True,
            hover_data=["trial_id", "rung", "target_epochs", "val_acc1", "lr_text", "weight_decay_text", "cumulative_flops"],
            title=f"{arch_title}: best validation accuracy during HPO",
        )
        fig_progress.update_xaxes(title="Used FLOPs budget, %")
        fig_progress.update_yaxes(title="Best validation Acc@1 so far")
        fig_progress.update_layout(height=430)
        fig_progress.show()

        fig_stages = px.scatter(
            arch_df,
            x="target_epochs",
            y="val_acc1",
            color="trial_label",
            symbol="rung",
            size="incremental_flops",
            hover_data=["budget_used_pct", "lr_text", "weight_decay_text", "cumulative_flops"],
            title=f"{arch_title}: all evaluated ASHA stages",
        )
        fig_stages.update_xaxes(title="Training epochs reached by trial")
        fig_stages.update_yaxes(title="Validation Acc@1")
        fig_stages.update_layout(height=500, legend_title_text="Trial")
        fig_stages.show()

        best_by_trial = (
            arch_df.sort_values(["trial_id", "val_acc1"])
            .groupby("trial_id", as_index=False)
            .tail(1)
            .sort_values("val_acc1", ascending=False)
            .copy()
        )
        fig_trials = px.bar(
            best_by_trial,
            x="trial_label",
            y="val_acc1",
            color="target_epochs",
            text="val_acc1",
            hover_data=["lr_text", "weight_decay_text", "rung", "budget_used_pct"],
            title=f"{arch_title}: best observed stage per trial",
            color_continuous_scale="Teal",
        )
        fig_trials.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig_trials.update_xaxes(title="Trial")
        fig_trials.update_yaxes(title="Best validation Acc@1", rangemode="tozero")
        fig_trials.update_layout(height=480)
        fig_trials.show()

        fig_hparams = px.scatter(
            best_by_trial,
            x="lr",
            y="weight_decay",
            color="val_acc1",
            size="target_epochs",
            text="trial_id",
            hover_data=["trial_label", "lr_text", "weight_decay_text", "target_epochs", "budget_used_pct"],
            title=f"{arch_title}: sampled hyperparameters and trial quality",
            color_continuous_scale="Viridis",
        )
        fig_hparams.update_xaxes(title="Learning rate", type="log")
        fig_hparams.update_yaxes(title="Weight decay", type="log")
        fig_hparams.update_traces(textposition="top center")
        fig_hparams.update_layout(height=520)
        fig_hparams.show()

Architecture row=0, index=13197: best Acc@1=86.40, trial=11, epochs=81, lr=6.96e-03, wd=2.93e-06


In [43]:
!zip -r hpo_output.zip hpo_output

  adding: hpo_output/ (stored 0%)
  adding: hpo_output/checkpoints/ (stored 0%)
  adding: hpo_output/checkpoints/arch_00_trial_11_epoch_0027.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_08_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_11_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_01_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_00.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_11.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_09_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_01.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_05_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_03_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_06_epoch_0003.pt (deflated 10%)
  adding: hpo_output/checkpoints/arch_00_trial_03.pt (deflated 10%)
  adding: hpo_ou